<a href="https://colab.research.google.com/github/santoshmanushree/First-mini-python-project/blob/main/Employee_Payroll_Dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import ipywidgets as widgets
import matplotlib.pyplot as plt

from IPython.display import display, clear_output, HTML
from datetime import date
from google.colab import files

print("Employee Payroll System Ready!")

Employee Payroll System Ready!


In [ ]:

# ==========================================
# EMPLOYEE PAYROLL MANAGEMENT SYSTEM
# Google Colab + Python
# ==========================================

import pandas as pd
import ipywidgets as widgets
import matplotlib.pyplot as plt

from IPython.display import display, clear_output, HTML
from datetime import date
from google.colab import files

# DATA STORAGE
employees = []
payroll_records = []

# COLORS AND STYLES
style = {"description_width": "120px"}
wide = widgets.Layout(width="95%")

# OUTPUT AREAS
dashboard_output = widgets.Output()
employee_output = widgets.Output()
payroll_output = widgets.Output()
report_output = widgets.Output()
slip_output = widgets.Output()

# INPUT FIELDS
emp_id = widgets.Text(description="Employee ID:", style=style, layout=wide)
emp_name = widgets.Text(description="Name:", style=style, layout=wide)
department = widgets.Dropdown(
    options=["HR", "IT", "Finance", "Marketing", "Sales", "Operations"],
    description="Department:", style=style, layout=wide
)
designation = widgets.Text(description="Designation:", style=style, layout=wide)
basic_salary = widgets.FloatText(
    description="Basic Salary:", style=style, layout=wide, value=0
)

# EMPLOYEE SELECTION
employee_select = widgets.Dropdown(
    options=[], description="Select Employee:", style=style, layout=wide
)

# OUTPUT HELPER
def show_message(output, message):
    with output:
        clear_output(wait=True)
        display(HTML(message))

# REFRESH EMPLOYEE DROPDOWN
def refresh_dropdown():
    employee_select.options = [
        (f"{e['ID']} - {e['Name']}", e['ID'])
        for e in employees
    ]

# DASHBOARD
def show_dashboard():
    with dashboard_output:
        clear_output(wait=True)

        total_employees = len(employees)
        total_payroll = sum(p["Net Salary"] for p in payroll_records)
        departments = len(set(e["Department"] for e in employees))

        display(HTML(f"""
        <div style="background:#17365D;color:white;padding:22px;
        border-radius:12px;text-align:center;">
        <h1>Employee Payroll Dashboard</h1>
        <p>Employee and Salary Management System</p>
        </div>
        <br>
        <div style="display:flex;gap:12px;flex-wrap:wrap;">
        <div style="background:#DCEBFF;padding:20px;border-radius:10px;
        flex:1;min-width:130px;">
        <h3>Total Employees</h3><h1>{total_employees}</h1></div>
        <div style="background:#DDF4E5;padding:20px;border-radius:10px;
        flex:1;min-width:130px;">
        <h3>Total Payroll</h3><h1>₹{total_payroll:,.2f}</h1></div>
        <div style="background:#FFF0D8;padding:20px;border-radius:10px;
        flex:1;min-width:130px;">
        <h3>Departments</h3><h1>{departments}</h1></div>
        <div style="background:#EEE3FF;padding:20px;border-radius:10px;
        flex:1;min-width:130px;">
        <h3>Salary Slips</h3><h1>{len(payroll_records)}</h1></div>
        </div>
        """))

        if employees:
            display(HTML("<h3>Recent Employee Records</h3>"))
            display(pd.DataFrame(employees).tail(5))

# ADD EMPLOYEE
def add_employee(b):
    if not all([emp_id.value.strip(), emp_name.value.strip(),
                designation.value.strip()]):
        show_message(employee_output,
                     "<h3 style='color:red'>Please fill all employee fields.</h3>")
        return

    if any(e["ID"] == emp_id.value.strip() for e in employees):
        show_message(employee_output,
                     "<h3 style='color:red'>Employee ID already exists.</h3>")
        return

    if basic_salary.value <= 0:
        show_message(employee_output,
                     "<h3 style='color:red'>Enter a positive salary.</h3>")
        return

    employees.append({
        "ID": emp_id.value.strip(),
        "Name": emp_name.value.strip(),
        "Department": department.value,
        "Designation": designation.value.strip(),
        "Basic Salary": basic_salary.value
    })

    refresh_dropdown()
    show_dashboard()

    show_message(employee_output,
                 "<h3 style='color:green'>Employee added successfully!</h3>")

    emp_id.value = ""
    emp_name.value = ""
    designation.value = ""
    basic_salary.value = 0

add_button = widgets.Button(
    description="Add Employee",
    button_style="success",
    icon="user-plus"
)
add_button.on_click(add_employee)

add_panel = widgets.VBox([
    widgets.HTML("<h2>Add New Employee</h2>"),
    emp_id, emp_name, department, designation, basic_salary,
    add_button, employee_output
])

# VIEW EMPLOYEES
def view_employees(b):
    with employee_output:
        clear_output(wait=True)
        if employees:
            display(pd.DataFrame(employees))
        else:
            print("No employee records found.")

view_button = widgets.Button(
    description="View Employees",
    button_style="info",
    icon="users"
)
view_button.on_click(view_employees)

# DELETE EMPLOYEE
def delete_employee(b):
    if not employee_select.value:
        show_message(employee_output, "No employee selected.")
        return

    selected_id = employee_select.value

    employees[:] = [e for e in employees if e["ID"] != selected_id]

    # Keep previous payroll records as historical records.
    refresh_dropdown()
    show_dashboard()

    show_message(employee_output,
                 "<h3 style='color:green'>Employee deleted successfully!</h3>")

delete_button = widgets.Button(
    description="Delete Selected Employee",
    button_style="danger",
    icon="trash"
)
delete_button.on_click(delete_employee)

# PAYROLL CALCULATION
def calculate_payroll(b):
    selected_id = employee_select.value

    if not selected_id:
        show_message(payroll_output, "Please select an employee.")
        return

    employee = next(
        (e for e in employees if e["ID"] == selected_id), None
    )

    if employee is None:
        show_message(payroll_output, "Employee not found.")
        return

    basic = employee["Basic Salary"]
    allowance = basic * 0.20
    deduction = basic * 0.10
    net_salary = basic + allowance - deduction

    record = {
        "Employee ID": employee["ID"],
        "Name": employee["Name"],
        "Date": str(date.today()),
        "Basic Salary": basic,
        "Allowance": allowance,
        "Deduction": deduction,
        "Net Salary": net_salary
    }

    payroll_records.append(record)

    with payroll_output:
        clear_output(wait=True)

        display(HTML(f"""
        <div style="border:2px solid #17365D;padding:25px;
        border-radius:12px;max-width:500px;">
        <h2 style="text-align:center;color:#17365D">
        EMPLOYEE SALARY SLIP</h2>
        <hr>
        <p><b>Employee ID:</b> {employee['ID']}</p>
        <p><b>Name:</b> {employee['Name']}</p>
        <p><b>Department:</b> {employee['Department']}</p>
        <p><b>Date:</b> {date.today()}</p>
        <hr>
        <p>Basic Salary: ₹{basic:,.2f}</p>
        <p>Allowance (20%): ₹{allowance:,.2f}</p>
        <p>Deduction (10%): ₹{deduction:,.2f}</p>
        <hr>
        <h2 style="color:green">Net Salary: ₹{net_salary:,.2f}</h2>
        </div>
        """))

    show_dashboard()

payroll_button = widgets.Button(
    description="Calculate Salary",
    button_style="success",
    icon="calculator"
)
payroll_button.on_click(calculate_payroll)

payroll_panel = widgets.VBox([
    widgets.HTML("<h2>Payroll Management</h2>"),
    employee_select,
    payroll_button,
    payroll_output
])

# PAYROLL HISTORY
def view_payroll(b):
    with slip_output:
        clear_output(wait=True)
        if payroll_records:
            display(pd.DataFrame(payroll_records))
        else:
            print("No payroll records available.")

history_button = widgets.Button(
    description="View Payroll History",
    button_style="info"
)
history_button.on_click(view_payroll)

# REPORTS AND CHARTS
def generate_report(b):
    with report_output:
        clear_output(wait=True)

        if not payroll_records:
            print("Calculate at least one salary to generate a report.")
            return

        df = pd.DataFrame(payroll_records)

        display(HTML("<h3>Payroll Summary Report</h3>"))
        display(df)

        department_totals = {}

        for record in payroll_records:
            emp = next(
                (e for e in employees
                 if e["ID"] == record["Employee ID"]), None
            )
            dept = emp["Department"] if emp else "Former Employee"
            department_totals[dept] = (
                department_totals.get(dept, 0) + record["Net Salary"]
            )

        plt.figure(figsize=(8, 4))
        plt.bar(department_totals.keys(), department_totals.values())
        plt.title("Department-wise Payroll")
        plt.xlabel("Department")
        plt.ylabel("Net Salary (₹)")
        plt.xticks(rotation=25)
        plt.tight_layout()
        plt.show()

report_button = widgets.Button(
    description="Generate Reports",
    button_style="info",
    icon="chart-bar"
)
report_button.on_click(generate_report)

# EXPORT RECORDS
def export_records(b):
    if not employees and not payroll_records:
        show_message(report_output, "No records available to export.")
        return

    if employees:
        pd.DataFrame(employees).to_csv(
            "employee_records.csv", index=False
        )
        files.download("employee_records.csv")

    if payroll_records:
        pd.DataFrame(payroll_records).to_csv(
            "payroll_records.csv", index=False
        )
        files.download("payroll_records.csv")

export_button = widgets.Button(
    description="Download CSV Records",
    button_style="success",
    icon="download"
)
export_button.on_click(export_records)

report_panel = widgets.VBox([
    widgets.HTML("<h2>Reports and Analytics</h2>"),
    report_button,
    export_button,
    report_output
])

# MAIN TABS
tabs = widgets.Tab(children=[
    dashboard_output,
    add_panel,
    widgets.VBox([
        widgets.HTML("<h2>Employee Management</h2>"),
        employee_select,
        view_button,
        delete_button,
        employee_output
    ]),
    payroll_panel,
    widgets.VBox([
        widgets.HTML("<h2>Payroll History</h2>"),
        history_button,
        slip_output
    ]),
    report_panel
])

tab_names = [
    "Dashboard",
    "Add Employee",
    "Manage Employees",
    "Payroll",
    "Salary Slips",
    "Reports"
]

for i, name in enumerate(tab_names):
    tabs.set_title(i, name)

show_dashboard()

display(tabs)
